# Mini-TP 1 — Del modelo al servicio (Sesión 1)

Sirve el modelo **Iris / RandomForest** por **REST** con FastAPI: contrato
Pydantic, `POST /v1/predict` versionado y `GET /health`.

El código del servicio vive en `src/model_service/` (`model.py` + `rest.py`),
compartido con el Mini-TP 2. Este notebook lo levanta y lo ejercita.

## 1. Entrenar el modelo (si falta el artefacto)

In [1]:
from pathlib import Path
import subprocess, sys

if not Path("../model/model.joblib").exists():
    subprocess.run([sys.executable, "../train.py"], check=True)
else:
    print("Artefacto ya presente:", Path("../model/model.joblib").resolve())

Artefacto ya presente: /home/chris/Documents/MIA/mlops-lse-fiuba-tps/model/model.joblib


## 2. Levantar la API REST en segundo plano

In [2]:
import threading, time, socket, urllib.request, uvicorn

def serve(app, port):
    def _run():
        uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=port,
                                      log_level="warning")).run()
    threading.Thread(target=_run, daemon=True).start()
    for _ in range(50):
        try:
            with socket.create_connection(("127.0.0.1", port), timeout=0.2):
                return
        except OSError:
            time.sleep(0.2)
    raise RuntimeError(f"el servidor en :{port} no respondió")

from model_service.rest import app
serve(app, 8000)
BASE = "http://127.0.0.1:8000"
print("REST arriba en", BASE, " (docs: /docs)")

REST arriba en http://127.0.0.1:8000  (docs: /docs)


## 3. `GET /health`

In [3]:
import requests
requests.get(f"{BASE}/health").json()

{'status': 'ok', 'model_loaded': True, 'model_version': '1.0.0'}

## 4. `POST /v1/predict` — caso válido

In [4]:
ok = {"sepal_length_cm": 5.1, "sepal_width_cm": 3.5,
      "petal_length_cm": 1.4, "petal_width_cm": 0.2}
r = requests.post(f"{BASE}/v1/predict", json=ok)
print("HTTP", r.status_code)
r.json()

HTTP 200


{'prediction': 0, 'prediction_label': 'setosa', 'model_version': '1.0.0'}

## 5. `POST /v1/predict` — caso inválido (se espera HTTP 422)

In [5]:
bad = {"sepal_length_cm": 5.1, "sepal_width_cm": 3.5,
       "petal_length_cm": 1.4, "petal_width_cm": -1.0}
r = requests.post(f"{BASE}/v1/predict", json=bad)
print("HTTP", r.status_code)
assert r.status_code == 422, "Se esperaba 422 para la entrada inválida"
r.json()

HTTP 422


{'detail': [{'type': 'greater_than',
   'loc': ['body', 'petal_width_cm'],
   'msg': 'Input should be greater than 0',
   'input': -1.0,
   'ctx': {'gt': 0.0}}]}

## 6. Conclusión

- El modelo queda detrás de un contrato **tipado**: entradas fuera de rango,
  tipos incorrectos o campos desconocidos se rechazan con **422** antes de
  llegar al modelo (`gt=0`, `extra="forbid"`).
- La respuesta incluye la **versión del modelo**, condición para trazabilidad.
- `/health` permite distinguir *servicio arriba* de *modelo cargado*.